# KNN com seleção de atributos usando o conteúdo das aulas

O primeiro exemplo da aula usa todos os atributos escolhidos para calcular as distâncias. Agora investigamos quais medidas ajudam a encontrar vizinhos da mesma classe.

A aula 2 apresenta seleção de subconjuntos de atributos nas páginas 78 a 81. Vamos comparar os 511 subconjuntos possíveis dos nove atributos numéricos, duas normalizações e duas distâncias. A previsão continua sendo a classe mais frequente entre K vizinhos.

Usamos NumPy para fazer as mesmas contas em vários exemplos de uma vez. O modelo segue implementado do zero. As saídas estão salvas para leitura, e a reexecução exige um kernel com NumPy.

### Resultado desta busca

Testamos 32.704 combinações de subconjuntos numéricos, escalas, distâncias e K. Em seguida, comparamos mais 16.384 combinações com atributos categóricos binarizados. Todas as previsões usam a classe majoritária, e as distâncias usam o mesmo peso por coluna.

A configuração escolhida usou glicose, hemoglobina e triglicerídeos, normalização pelo mínimo e máximo, Manhattan e K igual a 9. As categorias não melhoraram a configuração escolhida nessa comparação.

| Avaliação | Referência de aula com nove medidas | Seleção de atributos |
|---|---:|---:|
| Acurácia média, semente 42 | 72,34% | 86,06% |
| Acurácia média, semente 73 | 70,19% | 85,10% |
| Acurácia nas 104 amostras reservadas | 64,42% | 84,62% |
| F1 macro nas 104 amostras reservadas | 64,02% | 84,56% |
| Acertos nas 104 amostras reservadas | 67 | 88 |

A meta de 90% ainda não foi atingida nessas avaliações. Na reserva, a candidata acertou 41 dos 50 casos de controle e 47 dos 54 casos de diabetes. As referências de aula e as decisões sobre a base estão identificadas abaixo.

In [1]:
from pathlib import Path
from itertools import combinations
from statistics import mean
from time import perf_counter
import csv
import hashlib
import json
import random
import sys

import numpy as np

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent
caminho = raiz / "dados" / "treinamento_padronizado.csv"
with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

atributos = (
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC",
    "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
)
pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("Precisamos de um registro por paciente na divisão.")

classes = np.array([int(a["class"]) for a in amostras])
if not np.all(np.isin(classes, [0, 1])):
    raise ValueError("Esperávamos classes 0 e 1.")


def ler_numero(a, coluna):
    try:
        valor = float(a[coluna].strip().replace(",", "."))
    except ValueError:
        return np.nan
    if not np.isfinite(valor) or valor <= 0:
        return np.nan
    if coluna == "HEMOGLOBIN" and valor > 100:
        return np.nan
    return valor


numeros = np.array([[ler_numero(a, c) for c in atributos] for a in amostras])
controle = np.flatnonzero(classes == 0).tolist()
diabetes = np.flatnonzero(classes == 1).tolist()
gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)
n0, n1 = round(len(controle) * 0.20), round(len(diabetes) * 0.20)
reservadas = np.array(controle[:n0] + diabetes[:n1])
treinamento = np.array(controle[n0:] + diabetes[n1:])

print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")
print(f"Treinamento: {len(treinamento)} | Reservadas: {len(reservadas)}")
print("SHA-256 da base:", hashlib.sha256(caminho.read_bytes()).hexdigest())

Python 3.12.14 | NumPy 2.3.5
Treinamento: 416 | Reservadas: 104
SHA-256 da base: bc1a686d83c06e2d7667a6089e82b54b4bff2ca6467b81155f1246ac4719a032


### As comparações

Cada subconjunto tem de um a nove atributos. Testamos a distância Euclidiana e a Manhattan, com os atributos normalizados pelo mínimo e máximo ou pela média e desvio padrão. Todos os atributos da versão têm o mesmo peso na distância.

Comparamos K igual a 1, 3, 5, até 31. Todos são ímpares, então a votação entre duas classes sempre terá uma maioria. A seleção usa o maior F1 macro médio. Em caso de empate exato, preferimos menos atributos, menor K e a ordem fixa das normalizações e distâncias.

In [2]:
subconjuntos = [
    conjunto for tamanho in range(1, len(atributos) + 1)
    for conjunto in combinations(range(len(atributos)), tamanho)
]
escalas = ("minmax", "desvio")
distancias = ("euclidiana", "manhattan")
valores_k = np.arange(1, 32, 2)
configuracoes = [
    (escala, distancia, conjunto)
    for escala in escalas
    for distancia in distancias
    for conjunto in subconjuntos
]
tamanho_grade = (len(configuracoes), len(valores_k))
total_configuracoes = int(np.prod(tamanho_grade))
print(f"{len(subconjuntos)} subconjuntos × 2 escalas × 2 distâncias × {len(valores_k)} valores de K")
print(f"Configurações completas: {total_configuracoes}")

511 subconjuntos × 2 escalas × 2 distâncias × 16 valores de K
Configurações completas: 32704


### Preparação dentro de cada divisão

A mediana de cada coluna é calculada com os valores disponíveis na referência e preenche os campos ausentes. O mínimo, o máximo, a média e o desvio padrão também usam apenas os valores disponíveis dessa referência.

Na normalização pelo mínimo e máximo, subtraímos o mínimo e dividimos pela amplitude. Na outra, subtraímos a média e dividimos pelo desvio padrão amostral. Se a coluna não variar, ela recebe zero na divisão. A fórmula da distância usa somente as colunas escolhidas no subconjunto.

As distâncias iguais preservam uma ordem de referência embaralhada com semente 42.

In [3]:
def dividir(indices, partes, semente):
    grupos = [[] for _ in range(partes)]
    gerador = random.Random(semente)
    for classe in (0, 1):
        grupo = [int(i) for i in indices if classes[i] == classe]
        gerador.shuffle(grupo)
        for posicao, i in enumerate(grupo):
            grupos[posicao % partes].append(i)
    for posicao, grupo in enumerate(grupos):
        referencia = [i for j, outro in enumerate(grupos) if j != posicao for i in outro]
        yield np.array(referencia), np.array(grupo)


def normalizar(referencia, consulta, escala, preenchimento="mediana"):
    if preenchimento not in {"mediana", "media"}:
        raise ValueError("O preenchimento deve ser mediana ou media.")
    ref = numeros[referencia].copy()
    aval = numeros[consulta].copy()
    for j in range(len(atributos)):
        presentes = ref[np.isfinite(ref[:, j]), j]
        if not len(presentes):
            raise ValueError(f"Nenhuma medida disponível para {atributos[j]}.")
        estimativa = np.median(presentes) if preenchimento == "mediana" else np.mean(presentes)
        ref[np.isnan(ref[:, j]), j] = estimativa
        aval[np.isnan(aval[:, j]), j] = estimativa
        if escala == "minmax":
            centro = np.min(presentes)
            divisor = np.ptp(presentes)
        else:
            centro = np.mean(presentes)
            divisor = np.std(presentes, ddof=1) if len(presentes) > 1 else 0
        if divisor == 0:
            ref[:, j] = 0
            aval[:, j] = 0
        else:
            ref[:, j] = (ref[:, j] - centro) / divisor
            aval[:, j] = (aval[:, j] - centro) / divisor
    return ref, aval


def contar(reais, previstas):
    reais = reais.astype(bool)
    return np.stack((
        np.sum(~previstas & ~reais, axis=-1),
        np.sum(previstas & ~reais, axis=-1),
        np.sum(~previstas & reais, axis=-1),
        np.sum(previstas & reais, axis=-1),
    ), axis=-1).astype(np.int16)


def dividir_seguro(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def metricas(contagens):
    tn, fp, fn, tp = np.moveaxis(contagens, -1, 0)
    return {
        "acuracia": dividir_seguro(tn + tp, tn + fp + fn + tp),
        "f1": (
            dividir_seguro(2 * tn, 2 * tn + fp + fn)
            + dividir_seguro(2 * tp, 2 * tp + fp + fn)
        ) / 2,
        "precisao": (dividir_seguro(tn, tn + fn) + dividir_seguro(tp, tp + fp)) / 2,
        "recall0": dividir_seguro(tn, tn + fp),
        "recall1": dividir_seguro(tp, tp + fn),
    }


def avaliar_divisao(referencia, consulta, selecionadas=None, preenchimento="mediana"):
    referencia = referencia.tolist()
    random.Random(42).shuffle(referencia)
    referencia = np.array(referencia)
    if len(referencia) < max(valores_k):
        raise ValueError("O maior K precisa caber na referência.")

    cache = {}
    contagens = []
    if selecionadas is None:
        selecionadas = range(len(configuracoes))
    for indice in selecionadas:
        escala, distancia, conjunto = configuracoes[indice]
        if (escala, distancia) not in cache:
            ref, aval = normalizar(referencia, consulta, escala, preenchimento)
            diferencas = np.abs(aval[:, None, :] - ref[None, :, :])
            if distancia == "euclidiana":
                diferencas **= 2
            cache[escala, distancia] = np.moveaxis(diferencas, -1, 0)

        dist = cache[escala, distancia][list(conjunto)].sum(axis=0)
        if distancia == "euclidiana":
            dist = np.sqrt(dist)
        ordem = np.argsort(dist, axis=1, kind="stable")[:, :max(valores_k)]
        vizinhos = classes[referencia][ordem]
        votos_diabetes = np.cumsum(vizinhos, axis=1)[:, valores_k - 1]
        previstas = votos_diabetes > (valores_k / 2)
        contagens.append(contar(classes[consulta], previstas.T))
    return np.array(contagens)


def ordenar_resultados(rodadas):
    f1 = metricas(rodadas)["f1"].mean(axis=0).ravel()
    quantidades = np.repeat([len(c[2]) for c in configuracoes], len(valores_k))
    ks = np.tile(valores_k, len(configuracoes))
    return np.lexsort((np.arange(f1.size), ks, quantidades, -f1))


def localizar(posicao):
    i, j = np.unravel_index(posicao, tamanho_grade)
    return int(i), int(j)


def descrever(posicao):
    i, j = localizar(posicao)
    escala, distancia, conjunto = configuracoes[i]
    return f"{escala} | {distancia} | K={valores_k[j]} | " + ", ".join(atributos[c] for c in conjunto)


indice_base = configuracoes.index(("minmax", "euclidiana", tuple(range(9))))
j_base = list(valores_k).index(11)

### Seleção nas cinco rodadas

Usamos cinco partes com semente 42. Em cada rodada, quatro partes viram a referência e a quinta recebe as previsões. Repetimos isso para cada configuração e escolhemos pela maior média de F1 macro. Como essas pontuações participam da escolha, a maior delas descreve o resultado dentro da busca.

In [4]:
inicio = perf_counter()
rodadas = []
for numero, (ref, aval) in enumerate(dividir(treinamento, 5, 42), start=1):
    rodadas.append(avaliar_divisao(ref, aval))
    print(f"Rodada {numero} concluída")
busca = np.array(rodadas)
ordem = ordenar_resultados(busca)
escolhida = int(ordem[0])
i_final, j_final = localizar(escolhida)
print(f"Tempo da busca: {perf_counter() - inicio:.1f} s")

Rodada 1 concluída
Rodada 2 concluída
Rodada 3 concluída
Rodada 4 concluída
Rodada 5 concluída
Tempo da busca: 14.8 s


In [5]:
def mostrar(nome, contagens):
    m = metricas(contagens)
    print(f"{nome:22} | " + " | ".join(
        f"{np.mean(m[c]):>8.2%}" for c in ("acuracia", "f1", "precisao", "recall0", "recall1")
    ))


medias_f1 = metricas(busca)["f1"].mean(axis=0).ravel()
medias_acc = metricas(busca)["acuracia"].mean(axis=0).ravel()
for posicao in ordem[:10]:
    print(f"Acc {medias_acc[posicao]:.2%} | F1 {medias_f1[posicao]:.2%} | {descrever(posicao)}")

Acc 86.06% | F1 86.05% | minmax | manhattan | K=9 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 85.82% | F1 85.80% | minmax | manhattan | K=11 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 85.35% | F1 85.34% | minmax | manhattan | K=3 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 85.09% | F1 85.06% | minmax | manhattan | K=13 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 84.85% | F1 84.83% | minmax | euclidiana | K=15 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 84.86% | F1 84.83% | desvio | manhattan | K=9 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 84.85% | F1 84.82% | desvio | euclidiana | K=9 | GLUCOSE, HEMOGLOBIN
Acc 84.84% | F1 84.78% | minmax | manhattan | K=3 | GLUCOSE, HEMOGLOBIN
Acc 84.63% | F1 84.61% | desvio | manhattan | K=3 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 84.62% | F1 84.61% | minmax | euclidiana | K=3 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES


### Mesma candidata em outra divisão

Fixamos a candidata da semente 42 e repetimos a validação cruzada com semente 73. Comparamos com a referência da aula. Como as mesmas 416 amostras foram usadas na escolha anterior, essa repetição mede a sensibilidade à divisão.

Depois fazemos a comparação local nas mesmas 104 amostras reservadas, que já foram consultadas em outros experimentos.

In [6]:
outra_divisao = []
for ref, aval in dividir(treinamento, 5, 73):
    resultado = avaliar_divisao(ref, aval, [indice_base, i_final])
    outra_divisao.append([resultado[0, j_base], resultado[1, j_final]])
outra_divisao = np.array(outra_divisao)
print("Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
mostrar("Referência, semente 42", busca[:, indice_base, j_base])
mostrar("Candidata, semente 42", busca[:, i_final, j_final])
mostrar("Referência, semente 73", outra_divisao[:, 0])
mostrar("Candidata, semente 73", outra_divisao[:, 1])

Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Referência, semente 42 |   72.34% |   72.24% |   72.45% |   70.00% |   74.48%
Candidata, semente 42  |   86.06% |   86.05% |   86.26% |   90.00% |   82.40%
Referência, semente 73 |   70.19% |   70.10% |   70.18% |   67.50% |   72.67%
Candidata, semente 73  |   85.10% |   85.07% |   85.64% |   90.00% |   80.59%


In [7]:
resultado_reserva = avaliar_divisao(treinamento, reservadas, [indice_base, i_final])
base_reserva = resultado_reserva[0, j_base]
candidata_reserva = resultado_reserva[1, j_final]

print("Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
mostrar("Referência da aula", base_reserva)
mostrar("Candidata", candidata_reserva)
print("Matriz da referência:", base_reserva.reshape(2, 2).tolist())
print("Matriz da candidata:", candidata_reserva.reshape(2, 2).tolist())
print("Configuração escolhida:", descrever(escolhida))

Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Referência da aula     |   64.42% |   64.02% |   64.53% |   56.00% |   72.22%
Candidata              |   84.62% |   84.56% |   84.67% |   82.00% |   87.04%
Matriz da referência: [[28, 22], [15, 39]]
Matriz da candidata: [[41, 9], [7, 47]]
Configuração escolhida: minmax | manhattan | K=9 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES


### Registro das configurações e das métricas

Salvamos a grade, as contagens por divisão e os índices das combinações escolhidas. As contagens seguem a ordem TN, FP, FN e TP, com controle como classe 0. O código abaixo também permite recuperar as métricas de qualquer posição da grade.

In [8]:
caminho_registro = raiz / "resultados" / "knn_selecao_aulas.npz"
catalogo = {
    "atributos": atributos, "configuracoes": configuracoes,
    "valores_k": valores_k.tolist(), "semente_busca": 42, "semente_repeticao": 73,
    "sha256_base": hashlib.sha256(caminho.read_bytes()).hexdigest(),
}
np.savez_compressed(
    caminho_registro, catalogo=json.dumps(catalogo), busca=busca,
    outra_divisao=outra_divisao, escolhida=escolhida,
    reserva=np.array([base_reserva, candidata_reserva]),
)
print("Registro completo:", caminho_registro.relative_to(raiz))


def consultar(posicao):
    i, j = localizar(posicao)
    print(descrever(posicao))
    mostrar("Média das cinco rodadas", busca[:, i, j])
    for rodada, matriz in enumerate(busca[:, i, j], start=1):
        print(f"Rodada {rodada}: {matriz.reshape(2, 2).tolist()}")


consultar(escolhida)

Registro completo: resultados\knn_selecao_aulas.npz
minmax | manhattan | K=9 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Média das cinco rodadas |   86.06% |   86.05% |   86.26% |   90.00% |   82.40%
Rodada 1: [[35, 5], [7, 37]]
Rodada 2: [[37, 3], [6, 37]]
Rodada 3: [[34, 6], [11, 32]]
Rodada 4: [[38, 2], [7, 36]]
Rodada 5: [[36, 4], [7, 36]]


### Categorias como colunas de 0 e 1

A aula 2, página 91, apresenta binarização de atributos categóricos. Agora representamos cada categoria por uma coluna de 0 ou 1 e aplicamos a distância diretamente a todas as colunas, com o mesmo peso por coluna. As medidas numéricas continuam sendo normalizadas com a referência.

Testamos todas as 64 combinações de seis atributos categóricos, junto com quatro grupos numéricos. Esses grupos são os três exames da candidata, glicose e hemoglobina, os três exames com idade e as nove medidas originais. Comparamos as mesmas duas escalas, duas distâncias e 16 valores de K.

Definimos as categorias com o treinamento de cada rodada. Um campo desconhecido recebe uma categoria própria, e uma categoria preenchida que ainda não apareceu na referência recebe outra. Essas são decisões de representação para esta aplicação da binarização.

In [9]:
atributos_categoricos = ("SMOKER", "FAMILY_HISTORY", "PHYSICAL_ACTIVITY", "GENDER", "BLOOD_GROUP", "RH_FACTOR")
grupos_numericos = [
    (5, 6, 7), (5, 6), (0, 5, 6, 7), tuple(range(9))
]
grupos_categoricos = [
    grupo for tamanho in range(7)
    for grupo in combinations(range(6), tamanho)
]
configuracoes_categoricas = [
    (escala, distancia, grupo_num, grupo_cat)
    for escala in escalas for distancia in distancias
    for grupo_num in grupos_numericos for grupo_cat in grupos_categoricos
]


def diferencas_categoricas(referencia, consulta):
    diferencas = []
    for coluna in atributos_categoricos:
        def ler_categoria(a):
            valor = a[coluna].strip()
            if valor in {"", "desconhecido"} or (coluna == "GENDER" and valor == "N"):
                return "__desconhecido__"
            return valor

        ref = np.array([ler_categoria(amostras[i]) for i in referencia])
        aval = np.array([ler_categoria(amostras[i]) for i in consulta])
        aval = np.where(np.isin(aval, np.unique(ref)), aval, "__nao_visto__")
        diferencas.append(2.0 * (aval[:, None] != ref[None, :]))
    return np.array(diferencas)


def avaliar_categorias(referencia, consulta, indices=None):
    referencia = referencia.tolist()
    random.Random(42).shuffle(referencia)
    referencia = np.array(referencia)
    cat = diferencas_categoricas(referencia, consulta)
    cache = {}
    resultados = []
    if indices is None:
        indices = range(len(configuracoes_categoricas))

    for indice in indices:
        escala, distancia, grupo_num, grupo_cat = configuracoes_categoricas[indice]
        chave = (escala, distancia, grupo_num)
        if chave not in cache:
            ref, aval = normalizar(referencia, consulta, escala)
            dif = np.abs(aval[:, None, list(grupo_num)] - ref[None, :, list(grupo_num)])
            if distancia == "euclidiana":
                dif **= 2
            cache[chave] = dif.sum(axis=-1)

        d = cache[chave].copy()
        if grupo_cat:
            d += cat[list(grupo_cat)].sum(axis=0)
        if distancia == "euclidiana":
            d = np.sqrt(d)

        ordem = np.argsort(d, axis=1, kind="stable")[:, :max(valores_k)]
        vizinhos = classes[referencia][ordem]
        votos = np.cumsum(vizinhos, axis=1)[:, valores_k - 1]
        resultados.append(contar(classes[consulta], (votos > valores_k / 2).T))
    return np.array(resultados)


def descrever_categorias(i, j):
    escala, distancia, grupo_num, grupo_cat = configuracoes_categoricas[i]
    colunas = [atributos[c] for c in grupo_num] + [atributos_categoricos[c] for c in grupo_cat]
    return f"{escala} | {distancia} | K={valores_k[j]} | " + ", ".join(colunas)


busca_categorias = np.array([
    avaliar_categorias(ref, aval) for ref, aval in dividir(treinamento, 5, 42)
])
f1_cat = metricas(busca_categorias)["f1"].mean(axis=0).ravel()
quantidades_cat = np.repeat([
    len(c[2]) + len(c[3]) for c in configuracoes_categoricas
], len(valores_k))
ks_cat = np.tile(valores_k, len(configuracoes_categoricas))
ordem_cat = np.lexsort((np.arange(f1_cat.size), ks_cat, quantidades_cat, -f1_cat))
escolhida_cat = int(ordem_cat[0])
i_cat, j_cat = np.unravel_index(escolhida_cat, (len(configuracoes_categoricas), len(valores_k)))
i_cat, j_cat = int(i_cat), int(j_cat)

print("Combinações adicionais:", len(configuracoes_categoricas) * len(valores_k))
for posicao in ordem_cat[:5]:
    i, j = np.unravel_index(posicao, (len(configuracoes_categoricas), len(valores_k)))
    acuracia = metricas(busca_categorias[:, i, j])["acuracia"].mean()
    print(f"Acc {acuracia:.2%} | F1 {f1_cat[posicao]:.2%} | {descrever_categorias(i, j)}")

Combinações adicionais: 16384
Acc 86.06% | F1 86.05% | minmax | manhattan | K=9 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 85.82% | F1 85.80% | minmax | manhattan | K=11 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 85.35% | F1 85.34% | minmax | manhattan | K=3 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 85.09% | F1 85.06% | minmax | manhattan | K=13 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Acc 84.85% | F1 84.83% | minmax | euclidiana | K=15 | GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES


In [10]:
repeticao_cat = []
for ref, aval in dividir(treinamento, 5, 73):
    repeticao_cat.append(avaliar_categorias(ref, aval, [i_cat])[0, j_cat])
repeticao_cat = np.array(repeticao_cat)
reserva_cat = avaliar_categorias(treinamento, reservadas, [i_cat])[0, j_cat]

print("Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
mostrar("Busca numérica, 42", busca[:, i_final, j_final])
mostrar("Busca categorias, 42", busca_categorias[:, i_cat, j_cat])
mostrar("Categorias, 73", repeticao_cat)
mostrar("Categorias, reserva", reserva_cat)

with np.load(caminho_registro, allow_pickle=False) as registro:
    dados = {chave: registro[chave].copy() for chave in registro.files}
dados["categorias_configuracoes"] = json.dumps(configuracoes_categoricas)
dados["categorias_busca"] = busca_categorias
dados["categorias_escolhida"] = escolhida_cat
dados["categorias_repeticao"] = repeticao_cat
dados["categorias_reserva"] = reserva_cat
np.savez_compressed(caminho_registro, **dados)

Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Busca numérica, 42     |   86.06% |   86.05% |   86.26% |   90.00% |   82.40%
Busca categorias, 42   |   86.06% |   86.05% |   86.26% |   90.00% |   82.40%
Categorias, 73         |   85.10% |   85.07% |   85.64% |   90.00% |   80.59%
Categorias, reserva    |   84.62% |   84.56% |   84.67% |   82.00% |   87.04%


### Preencher os campos ausentes pela média ou pela mediana

A média é a soma dos valores disponíveis dividida pela quantidade deles. Por exemplo, se tivermos 80, 100 e 120, a média é 100. Podemos usar esse valor como estimativa para uma medida que está ausente.

Estimativa dos valores ausentes aparece na aula 2, páginas 44 a 47 e 52. Média e mediana são apresentadas na aula 4, página 9. A escolha entre as duas estimativas é uma decisão desta aplicação. O preenchimento é feito nos atributos numéricos. Categorias como tabagismo precisam de uma regra para categorias.

Mantemos os três exames, mínimo e máximo, Manhattan e K igual a 9. A única mudança é a estimativa do campo ausente. Calculamos tanto a média quanto a mediana com os valores disponíveis no treinamento de cada rodada. Os mínimos e máximos continuam sendo os valores observados nessa referência.

In [11]:
print("Estimativas nas 416 amostras de treinamento")
print("Atributo         | Ausentes | Média   | Mediana")
for j in configuracoes[i_final][2]:
    disponiveis = numeros[treinamento, j]
    ausentes = np.isnan(disponiveis).sum()
    disponiveis = disponiveis[np.isfinite(disponiveis)]
    print(
        f"{atributos[j]:16} | {ausentes:>8} | "
        f"{np.mean(disponiveis):>7.2f} | {np.median(disponiveis):>7.2f}"
    )

contagens_preenchimento = {}
for semente in (42, 73):
    for preenchimento in ("mediana", "media"):
        rodadas = []
        for ref, aval in dividir(treinamento, 5, semente):
            resultado = avaliar_divisao(
                ref, aval, [i_final], preenchimento=preenchimento
            )
            rodadas.append(resultado[0, j_final])
        contagens_preenchimento[semente, preenchimento] = np.array(rodadas)

print()
print("Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for (semente, preenchimento), contagens in contagens_preenchimento.items():
    mostrar(f"{preenchimento}, semente {semente}", contagens)

Estimativas nas 416 amostras de treinamento
Atributo         | Ausentes | Média   | Mediana
GLUCOSE          |       50 |  114.08 |  103.00
HEMOGLOBIN       |       97 |   11.00 |    6.40
TRIGLYCERIDES    |       38 |  183.66 |  151.00

Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
mediana, semente 42    |   86.06% |   86.05% |   86.26% |   90.00% |   82.40%
media, semente 42      |   84.86% |   84.82% |   85.67% |   91.00% |   79.16%
mediana, semente 73    |   85.10% |   85.07% |   85.64% |   90.00% |   80.59%
media, semente 73      |   83.90% |   83.86% |   84.56% |   89.50% |   78.73%


In [12]:
reserva_preenchimento = {}
for preenchimento in ("mediana", "media"):
    resultado = avaliar_divisao(
        treinamento, reservadas, [i_final], preenchimento=preenchimento
    )
    reserva_preenchimento[preenchimento] = resultado[0, j_final]

print("Nas 104 amostras reservadas")
print("Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for preenchimento, contagens in reserva_preenchimento.items():
    mostrar(preenchimento, contagens)
    print("Matriz:", contagens.reshape(2, 2).tolist())

with np.load(caminho_registro, allow_pickle=False) as registro:
    dados = {chave: registro[chave].copy() for chave in registro.files}
dados["preenchimento_ordem"] = json.dumps(list(contagens_preenchimento))
dados["preenchimento_rodadas"] = np.array(list(contagens_preenchimento.values()))
dados["preenchimento_reserva"] = np.array(list(reserva_preenchimento.values()))
np.savez_compressed(caminho_registro, **dados)

Nas 104 amostras reservadas
Versão                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
mediana                |   84.62% |   84.56% |   84.67% |   82.00% |   87.04%
Matriz: [[41, 9], [7, 47]]
media                  |   82.69% |   82.69% |   82.69% |   84.00% |   81.48%
Matriz: [[42, 8], [10, 44]]


#### O que encontramos

A média ficou acima da mediana nos três exames. Na hemoglobina, foi 11,00 contra 6,40. Isso mostra como as duas estimativas podem colocar uma amostra com exame ausente em posições diferentes na busca pelos vizinhos.

Com a média, a acurácia média caiu de 86,06% para 84,86% na semente 42 e de 85,10% para 83,90% na semente 73. Nas 104 amostras reservadas, acertamos 86 casos com a média e 88 com a mediana. Mantemos a mediana nesta configuração.
